In [1]:
from datasets import load_dataset

ds = load_dataset("SWE-bench/SWE-smith-trajectories", split="xml")

In [2]:
df = ds.to_polars()
len(df)

26076

In [3]:
df2 = df.select("messages", "resolved", "instance_id")

In [4]:
SEED = 42
MAX_EXAMPLES_PER_INSTANCE = 3

df3 = (
    df2
    .sample(fraction=1.0, shuffle=True, seed=SEED)
    .group_by("instance_id")
    .head(MAX_EXAMPLES_PER_INSTANCE)
)
len(df3)

20585

In [5]:
import polars as pl

LOW_QUALITY_INSTANCE_TYPES = [
    ".lm_modify",
    ".combine",
    ".func",
]

df4 = df3.filter(
    ~pl.col("instance_id").str.contains(instance_type)
    for instance_type in LOW_QUALITY_INSTANCE_TYPES
)
len(df4)

10700

In [6]:
df5 = df4.drop("instance_id")

In [ ]:
import json

RESOLVED_MESSAGE = "Your submission passed the repository tests, so the issue is resolved."
UNRESOLVED_MESSAGE = "Your submission failed the repository tests, so the issue remains unresolved."
RESTART_MESSSAGE = "The entire environment, including all files and state, has been restored to its exact state at the beginning of the task. You will now receive the same task again. Start from scratch, applying what you learned from this attempt."

def append_final_message(messages, resolved):
    messages = json.loads(messages)
    messages.append({
        "role": "user",
        "content": "\n".join([
            "OBSERVATION:",
            RESOLVED_MESSAGE if resolved else UNRESOLVED_MESSAGE,
            RESTART_MESSSAGE,
        ]),
    })
    messages = json.dumps(messages)
    return messages


df6 = (
    df5.with_columns(
        pl.struct("messages", "resolved")
        .map_elements(
            lambda row: append_final_message(row["messages"], row["resolved"]),
            return_dtype=pl.String
        )
        .alias("messages")
    )
)

In [8]:
TOKENS_PER_CHAR = 0.28
MODEL_MAX_LENGTH = 32768

df7 = df6.filter(pl.col("messages").str.len_chars() * TOKENS_PER_CHAR < MODEL_MAX_LENGTH / 2)
len(df7)

2682

In [9]:
from pathlib import Path

path = Path("data", "swe_smith.parquet")
path.parent.mkdir(parents=True, exist_ok=True)

df7.write_parquet(path)